<a href="https://colab.research.google.com/github/polaya813/polaya813/blob/main/1_Pract_CU_Regresion_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de uso de Regresión


# Caso de Uso: Predicción del Rendimiento Académico

## Enunciado

Una universidad desea construir un modelo que permita predecir la **nota final de un estudiante**
a partir de variables académicas y personales.

El dataset contiene:

- Horas de estudio por semana
- Porcentaje de asistencia
- Nivel de participación (1–10)
- Número de tareas entregadas
- Horas de trabajo semanal (empleo)
- Nota final

El dataset contiene intencionalmente:
- Valores nulos
- Registros duplicados
- Outliers

Propósito de la actividad: Generar un modelo de ML supervisado de regresión lineal.

BLOQUE 1. Importar Librerías

In [1]:
# Importamos las bibliotecas necesarias
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

BLOQUE 2: Preparar los Datos

In [5]:
import pandas as pd

# Leer datos desde el archivo csv y cargarlo en un dataframe

df = pd.read_csv('/content/1. dataset_rendimiento_academico_nvas_variables.csv')
# Visualizar el dataframe creado
print("Dimensiones del dataset:", df.shape)
display(df.head())
# print(df.head())

Dimensiones del dataset: (82, 9)


,horas_estudio_semana,porcentaje_asistencia,nivel_participacion,tareas_entregadas,horas_trabajo_semana,nota_final,genero,nivel_socioeconomico,modalidad
0,17.5,82.8,1,7,10.0,68.4,Femenino,Bajo,Presencial
1,14.3,88.6,4,11,14.5,58.0,Masculino,Alto,Hibrida
2,18.2,99.8,1,6,21.6,46.1,Femenino,Alto,Hibrida
3,22.6,79.8,5,6,12.5,72.1,Femenino,Medio,Virtual
4,NaN,76.9,4,11,13.4,54.1,Femenino,Bajo,Virtual


In [6]:
# Explorar los datos

print("\nResumen estadístico:")
display(df.describe())

print("\nValores nulos:")
print(df.isnull().sum())

print("\nDuplicados:", df.duplicated().sum())


Resumen estadístico:


,horas_estudio_semana,porcentaje_asistencia,nivel_participacion,tareas_entregadas,horas_trabajo_semana,nota_final
count,81.000000,81.000000,82.000000,82.000000,82.000000,81.000000
mean,15.006173,84.634568,5.073171,9.853659,10.973171,66.132099
std,7.008162,9.274092,2.934598,2.931826,5.965933,19.663887
min,1.900000,65.800000,1.000000,5.000000,-1.200000,33.500000
25%,11.800000,78.000000,3.000000,8.000000,7.000000,56.500000
50%,14.400000,85.100000,5.000000,10.000000,11.000000,64.000000
75%,18.100000,89.700000,7.750000,12.000000,14.500000,72.100000
max,60.000000,109.600000,10.000000,14.000000,29.500000,200.000000



Valores nulos:
horas_estudio_semana     1
porcentaje_asistencia    1
nivel_participacion      0
tareas_entregadas        0
horas_trabajo_semana     0
nota_final               1
genero                   0
nivel_socioeconomico     0
modalidad                0
dtype: int64

Duplicados: 0


In [7]:
# Depurar los datos

# 1. Eliminar duplicados
df = df.drop_duplicates()

# 2. Imputar nulos con la mediana
df.fillna(df.median(numeric_only=True), inplace=True)

# 3. Detección y eliminación de outliers usando IQR
# Seleccionar solo columnas numéricas
df_numerico = df.select_dtypes(include=['number'])

Q1 = df_numerico.quantile(0.25)
Q3 = df_numerico.quantile(0.75)
IQR = Q3 - Q1

outliers_mask = (
    (df_numerico < (Q1 - 1.5 * IQR)) |
    (df_numerico > (Q3 + 1.5 * IQR))
)

# df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)] # Elimina todas las filas que contienen outliers
df = df[~outliers_mask.any(axis=1)]

print("Dimensiones después de limpieza:", df.shape)

Dimensiones después de limpieza: (74, 9)


In [8]:
# Transformar datos

# Transformar variable género de valores categóricos a valores numericos (0, 1)
df['genero'] = df['genero'].map({
    'Femenino': 0,
    'Masculino': 1
})

# Transformar variable nivel socioeconómico de valores categóricos a valores numéricos jerarquicos (1, 2, 3)
df['nivel_socioeconomico'] = df['nivel_socioeconomico'].map({
    'Bajo': 0,
    'Medio': 1,
    'Alto': 2
})

# Transformar variable modalidad de valores categóricos (no ordinales) a valores numéricos no ordinales.
df = pd.get_dummies(df, columns=['modalidad'], drop_first=True)


In [ ]:
print(df.describe())

       horas_estudio_semana  porcentaje_asistencia  nivel_participacion  \
count             74.000000              74.000000            74.000000   
mean              14.520270              84.008108             4.824324   
std                4.607379               8.784728             2.882912   
min                5.100000              65.800000             1.000000   
25%               12.000000              77.375000             2.250000   
50%               14.500000              84.700000             5.000000   
75%               17.650000              89.525000             7.000000   
max               24.300000             103.900000            10.000000   

       tareas_entregadas  horas_trabajo_semana  nota_final     genero  \
count          74.000000             74.000000   74.000000  74.000000   
mean            9.851351             10.793243   63.936486   0.500000   
std             3.019037              5.739871   11.073151   0.503413   
min             5.000000        

In [ ]:
display(df.head())


,horas_estudio_semana,porcentaje_asistencia,nivel_participacion,tareas_entregadas,horas_trabajo_semana,nota_final,genero,nivel_socioeconomico,modalidad_Presencial,modalidad_Virtual
0,17.5,82.8,1,7,10.0,68.4,0,0,True,False
1,14.3,88.6,4,11,14.5,58.0,1,2,False,False
2,18.2,99.8,1,6,21.6,46.1,0,2,False,False
3,22.6,79.8,5,6,12.5,72.1,0,1,False,True
4,14.4,76.9,4,11,13.4,54.1,0,0,False,True


BLOQUE 3: Entrenar el Modelo

In [9]:
# Definir las variables de entrada (X) y la variable a predecir (y).
X = df.drop("nota_final", axis=1)
y = df["nota_final"]

# Dividimos los datos en entrenamiento (para que el modelo aprenda)
# y en prueba (para evaluar su precisión en datos que no ha visto).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Asignamos el modelo de regresión lineal
modelo_regresion = LinearRegression()

# Entrenar el modelo con el método .fit sobre los datos de entreanemiento (train) para generar la función predictora
modelo_regresion.fit(X_train, y_train)

# Utilizar la función o modelo predictor con los datos de pruebas (test)
y_pred = modelo_regresion.predict(X_test)

print("--- Resultados del Modelo ---")
# print("Notas finales reales (y_test): ", [f"{p:.2f}" for p in y_test])
# print("Notas finales predichas (y_pred):      ", [f"{p:.2f}" for p in y_pred])


--- Resultados del Modelo ---


BLOQUE 4: Evaluar el Modelo

In [10]:
# Evaluamos qué tan bien lo hizo el modelo.
# Determinar el (MSE), el (RMSE) y el R2.
mse = mean_squared_error(y_test, y_pred) # Equivale a (y_test, y_pred, squared=True)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MSE:", round(mse,2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2,2))


MSE: 40.16
RMSE: 6.34
R2: 0.68


BLOQUE 5: Usar el Modelo

In [11]:
print(X.columns)


Index(['horas_estudio_semana', 'porcentaje_asistencia', 'nivel_participacion',
       'tareas_entregadas', 'horas_trabajo_semana', 'genero',
       'nivel_socioeconomico', 'modalidad_Presencial', 'modalidad_Virtual'],
      dtype='object')


In [12]:
# Utilizar el modelo para realizar una predicción con datos nuevos de un estudiante.

nuevo_estudiante = pd.DataFrame([{
    "horas_estudio_semana": 20,
    "porcentaje_asistencia": 90,
    "nivel_participacion": 8,
    "tareas_entregadas": 12,
    "horas_trabajo_semana": 5,
    "genero": 1,                       # Masculino
    "nivel_socioeconomico": 2,         # Alto
    "modalidad_Presencial": 1,
    "modalidad_Virtual": 0
}])
# nuevo_estudiante = nuevo_estudiante.reindex(columns=X.columns, fill_value=0)
# print(X.columns)
nota_estimada = modelo_regresion.predict(nuevo_estudiante)

print("Nota estimada:", round(nota_estimada[0],2))

Nota estimada: 82.85


3. Conclusiones y Conexión con el Marco Teórico:

El código anterior es una implementación directa de la regresión.

pd.DataFrame(data) es la fase de datos, donde la variable 'precio' es el valor numérico que queremos predecir.

LinearRegression() es el modelo de regresión que seleccionamos para este tipo de problema.

modelo_regresion.fit() es el proceso de entrenamiento donde el modelo "aprende".

modelo_regresion.predict() es la fase de predicción sobre los datos nuevos.